# Vanguard Website A/B Test Analysis

## 1. Project overview

Vanguard compared its traditional online process (**Control**) with a redesigned
interface (**Test**) during the project period March 15–June 20, 2017. Both versions
use the sequence `start → step_1 → step_2 → step_3 → confirm`.

This analysis asks whether the redesign increases visit-level completion, how
backwards navigation and recorded elapsed time differ, and whether completion
improves by more than the required **5% relative uplift**.

This notebook retains the original client cleaning, consecutive-step compression,
backwards-step loop, and timing calculations. Completion is consistently binary
per visit within each experiment group. All intermediate data stay in memory.

## 2. Setup

Install `../requirements.txt` and run this notebook from the `notebooks/` directory.
The four raw files are included in the repository; loading requires no network access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

DATA_DIR = Path("../data/raw")

### Load the source datasets

Client profiles contain demographic, account, and recent activity fields. The two
web-event files contain timestamped process steps. The experiment roster assigns
clients to Test or Control; missing assignments are outside the experiment analysis.

In [2]:
clients_raw = pd.read_csv(DATA_DIR / "df_final_demo.txt")
web_part_1 = pd.read_csv(DATA_DIR / "df_final_web_data_pt_1.txt")
web_part_2 = pd.read_csv(DATA_DIR / "df_final_web_data_pt_2.txt")
web_events_raw = pd.concat([web_part_1, web_part_2], ignore_index=True)
experiment_raw = pd.read_csv(DATA_DIR / "df_final_experiment_clients.txt")
del web_part_1, web_part_2

pd.DataFrame({
    "dataset": ["Client profiles", "Web events (combined)", "Experiment roster"],
    "rows": [len(clients_raw), len(web_events_raw), len(experiment_raw)],
    "unique_clients": [clients_raw.client_id.nunique(),
                       web_events_raw.client_id.nunique(),
                       experiment_raw.client_id.nunique()],
}).set_index("dataset")

,rows,unique_clients
dataset,,
Client profiles,70609,70609
Web events (combined),755405,120157
Experiment roster,70609,70609


## 3. Data preparation

### Client profiles

In [3]:
pd.DataFrame({
    "dtype": clients_raw.dtypes.astype(str),
    "missing": clients_raw.isna().sum(),
    "unique": clients_raw.nunique(),
})

,dtype,missing,unique
client_id,int64,0,70609
clnt_tenure_yr,float64,14,54
clnt_tenure_mnth,float64,14,482
clnt_age,float64,15,165
gendr,object,14,4
num_accts,float64,14,8
bal,float64,14,70328
calls_6_mnth,float64,14,8
logons_6_mnth,float64,14,9


Retain the original rule requiring at least seven non-missing fields per client.
Fill the remaining missing age using the **mean**, then preserve the original
integer conversion (truncation, not rounding). Map gender `X` to `U` (unknown),
and use descriptive column names.

In [4]:
clients = clients_raw.dropna(thresh=7).copy()
clients_removed = len(clients_raw) - len(clients)
ages_imputed = int(clients["clnt_age"].isna().sum())
clients["clnt_age"] = clients["clnt_age"].fillna(clients["clnt_age"].mean())
clients["gendr"] = clients["gendr"].map({"X": "U", "F": "F", "M": "M", "U": "U"})
integer_columns = ["clnt_tenure_yr", "clnt_tenure_mnth", "clnt_age",
                   "num_accts", "calls_6_mnth", "logons_6_mnth"]
for column in integer_columns:
    clients[column] = clients[column].astype(int)
clients.columns = ["client_id", "client_tenure_yr", "client_tenure_month",
                   "client_age", "gender", "num_accounts", "balance",
                   "calls_6_month", "logons_6_month"]
assert clients.client_id.is_unique
print(f"Removed {clients_removed} incomplete profiles; imputed {ages_imputed} age value.")
clients.head()

Removed 14 incomplete profiles; imputed 1 age value.


,client_id,client_tenure_yr,client_tenure_month,client_age,gender,num_accounts,balance,calls_6_month,logons_6_month
0,836976,6,73,60,U,2,45105.30,6,9
1,2304905,7,94,58,U,2,110860.30,6,9
2,1439522,5,64,32,U,2,52467.79,6,9
3,1562045,16,198,49,M,2,67454.65,3,6
4,5126305,12,145,33,F,2,103671.75,0,3


### Experiment roster and population merge

In [5]:
experiment = experiment_raw.rename(columns={"Variation": "variation"}).copy()
assert experiment.client_id.is_unique
assert set(experiment.variation.dropna().unique()) == {"Control", "Test"}
experiment_clients = clients.merge(experiment, on="client_id", how="left", validate="one_to_one")
experiment_clients = experiment_clients.dropna(subset=["variation"]).copy()

population_counts = pd.concat([
    experiment.variation.value_counts().rename("roster_clients"),
    experiment_clients.variation.value_counts().rename("retained_clients"),
], axis=1).sort_index()
population_counts["excluded_incomplete_profiles"] = (
    population_counts.roster_clients - population_counts.retained_clients
)
population_counts

,roster_clients,retained_clients,excluded_incomplete_profiles
variation,,,
Control,23532,23527,5
Test,26968,26961,7


The performance population uses the retained experiment clients, matching the
original notebook. Removing incomplete demographic profiles therefore also removes
their web events. The roster counts and retained counts are shown separately.

### Web-event checks and chronological ordering

In [6]:
pd.DataFrame({
    "dtype": web_events_raw.dtypes.astype(str),
    "missing": web_events_raw.isna().sum(),
    "unique": web_events_raw.nunique(),
})

,dtype,missing,unique
client_id,int64,0,120157
visitor_id,object,0,130236
visit_id,object,0,158095
process_step,object,0,5
date_time,object,0,629363


In [7]:
web_events = web_events_raw[
    web_events_raw.client_id.isin(experiment_clients.client_id)
].copy()
web_events["date_time"] = pd.to_datetime(web_events["date_time"])
sequence = ["start", "step_1", "step_2", "step_3", "confirm"]
assert web_events[["visit_id", "process_step", "date_time"]].notna().all().all()
assert set(web_events.process_step.unique()) == set(sequence)
web_events = web_events.sort_values(["visit_id", "date_time"])
web_events.process_step.value_counts().reindex(sequence).rename("raw_experiment_events")

process_step
start      104046
step_1      68412
step_2      56857
step_3      48677
confirm     43215
Name: raw_experiment_events, dtype: int64

Preserve the original removal of consecutive repeated steps within each `visit_id`:
keep the first row of each run. There is no additional global deduplication or
outlier filtering. Equal timestamps retain their input order; their true sequence
cannot be established from timestamps alone.

In [8]:
repeated_step = (
    web_events.visit_id.eq(web_events.visit_id.shift())
    & web_events.process_step.eq(web_events.process_step.shift())
)
events = web_events.loc[~repeated_step].copy().reset_index(drop=True)
print(f"Retained {len(events):,} events after removing {int(repeated_step.sum()):,} consecutive repeats.")
events = events.merge(
    experiment[["client_id", "variation"]], on="client_id", how="left", validate="many_to_one"
)
assert events.variation.notna().all()
events.head()

Retained 284,503 events after removing 36,704 consecutive repeats.


,client_id,visitor_id,visit_id,process_step,date_time,variation
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,Test
1,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,Test
2,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,Test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,Test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:35,Test


In [9]:
visit_id_checks = pd.Series({
    "visit_ids_shared_by_multiple_clients": int((events.groupby("visit_id").client_id.nunique() > 1).sum()),
    "visit_ids_appearing_in_both_groups": int((events.groupby("visit_id").variation.nunique() > 1).sum()),
    "visit_timestamp_pairs_with_different_steps": int(
        (events.groupby(["visit_id", "date_time"]).process_step.nunique() > 1).sum()
    ),
})
visit_id_checks

visit_ids_shared_by_multiple_clients          235
visit_ids_appearing_in_both_groups            118
visit_timestamp_pairs_with_different_steps    369
dtype: int64

Some visit IDs are shared across clients and groups. To preserve the original
analysis, timing and backwards-step detection retain grouping by `visit_id`;
completion retains one row per `(variation, visit_id)`, as in the original z-test.
Shared IDs can mix client journeys and mean the groups are not fully independent.
No affected visits are silently excluded. This is a limitation of the current
analysis; resolving session identity would require a separately agreed analysis.

## 4. Client and experiment population analysis

The first table describes **all retained client profiles**. The subsequent tables
describe **retained experiment clients only**; these populations are different.

In [10]:
profile_columns = ["client_age", "client_tenure_yr", "num_accounts", "balance",
                   "calls_6_month", "logons_6_month"]
clients[profile_columns].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
client_age,70595.0,46.18,15.60,13.00,32.00,47.0,59.0,96.00
client_tenure_yr,70595.0,12.05,6.87,2.00,6.00,11.0,16.0,62.00
num_accounts,70595.0,2.26,0.53,1.00,2.00,2.0,2.0,8.00
balance,70595.0,147445.24,301508.71,13789.42,37346.84,63332.9,137544.9,16320040.15
calls_6_month,70595.0,3.38,2.24,0.00,1.00,3.0,6.0,7.00
logons_6_month,70595.0,5.57,2.35,1.00,4.00,5.0,7.0,9.00


In [11]:
experiment_clients[profile_columns].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
client_age,50488.0,47.06,15.53,17.00,33.00,48.0,59.00,96.00
client_tenure_yr,50488.0,12.03,6.86,2.00,6.00,11.0,16.00,55.00
num_accounts,50488.0,2.25,0.53,1.00,2.00,2.0,2.00,7.00
balance,50488.0,149514.68,302036.42,23789.44,39878.41,65733.6,139956.54,16320040.15
calls_6_month,50488.0,3.09,2.19,0.00,1.00,3.0,5.00,6.00
logons_6_month,50488.0,6.13,2.18,3.00,4.00,6.0,8.00,9.00


In [12]:
experiment_clients.groupby("variation")[profile_columns].mean().round(2)

,client_age,client_tenure_yr,num_accounts,balance,calls_6_month,logons_6_month
variation,,,,,,
Control,47.26,12.09,2.26,150147.33,3.13,6.17
Test,46.89,11.98,2.25,148962.61,3.06,6.10


Retain the original age bands and right-inclusive boundaries: 0–18, 19–34, 35–54,
55–74, and over 74, using the truncated integer ages. Unknown gender remains visible.

In [13]:
age_bins = [0, 18, 34, 54, 74, clients.client_age.max()]
age_labels = ["0–18", "19–34", "35–54", "55–74", ">74"]
experiment_clients["age_group"] = pd.cut(
    experiment_clients.client_age, bins=age_bins, labels=age_labels, include_lowest=True
)
pd.crosstab(experiment_clients.age_group, experiment_clients.variation)

variation,Control,Test
age_group,,
0–18,107,135
19–34,6251,7383
35–54,8540,9862
55–74,7952,8826
>74,677,755


In [14]:
pd.crosstab(experiment_clients.gender, experiment_clients.variation)

variation,Control,Test
gender,,
F,7543,8716
M,7970,8977
U,8014,9268


In [15]:
experiment_clients.groupby(["variation", "age_group"], observed=True).agg(
    clients=("client_id", "size"),
    mean_tenure_years=("client_tenure_yr", "mean"),
    mean_balance=("balance", "mean"),
    mean_calls_6_month=("calls_6_month", "mean"),
).round(2)

clients  mean_tenure_years  mean_balance  \
variation age_group                                             
Control   0–18           107               8.82      42231.36   
          19–34         6251               9.18      70511.58   
          35–54         8540              12.22     142941.17   
          55–74         7952              13.83     211965.00   
          >74            677              17.27     267306.19   
Test      0–18           135               8.96      42117.52   
          19–34         7383               9.16      70067.26   
          35–54         9862              12.15     142414.22   
          55–74         8826              13.75     216296.40   
          >74            755              17.25     237969.94   

                     mean_calls_6_month  
variation age_group                      
Control   0–18                     2.12  
          19–34                    3.21  
          35–54                    2.89  
          55–74                    3.30  
          >74                      3.57  
Test      0–18                     2.67  
          19–34                    3.08  
          35–54                    2.86  
          55–74                    3.25  
          >74                      3.38

The demographic tables describe who is represented and how the groups compare.
They are descriptive summaries, not a formal randomization or balance test.

## 5. Performance metrics

### 5.1 Recorded elapsed time

The original code calculates `current timestamp − previous timestamp` after
consecutive-step compression. It attaches this interval to the **arriving step**,
and assigns zero to the first row of each visit. The later notebook aggregates
these same values; no later alignment correction is present in the source notebook.

The values below are retained and accurately labelled **elapsed seconds since the
previous recorded step**. They are not estimates of time spent on the named page.
In particular, returning to `start` can have a nonzero interval, and final-page
dwell time is unobserved. No page-time correction from an external presentation
has been assumed.

In [16]:
events["elapsed_since_previous_seconds"] = (
    events.groupby("visit_id").date_time.diff().dt.total_seconds().fillna(0)
)
assert events.elapsed_since_previous_seconds.ge(0).all()

step_times = events.groupby(["visit_id", "process_step"])["elapsed_since_previous_seconds"].sum()
mean_step_totals = step_times.groupby("process_step").mean().reindex(sequence)
mean_step_totals.rename("mean_accumulated_incoming_seconds_per_visit_with_step").to_frame().round(2)

,mean_accumulated_incoming_seconds_per_visit_with_step
process_step,
start,49.25
step_1,79.23
step_2,54.86
step_3,103.40
confirm,109.09


The table above first sums incoming intervals for each visit and step, then averages
over visits containing that step (the original `avg_step_times` calculation).
The next table averages individual incoming intervals, preserving the original
Test/Control event-level timing summary. These denominators answer different questions.

In [17]:
incoming_time_by_group = events.groupby(["variation", "process_step"])[
    "elapsed_since_previous_seconds"
].mean().unstack("variation").reindex(sequence)
incoming_time_by_group.round(2).rename_axis("arriving_step")

variation,Control,Test
arriving_step,,
start,35.69,43.04
step_1,65.86,60.13
step_2,38.38,49.01
step_3,91.25,87.91
confirm,128.43,93.68


### 5.2 Backwards-step error rate

An error is a move from a later step to an earlier step in the sequence. Preserve
the original mapping and per-visit loop, marking the arriving row of a backwards
transition. The denominator is **all retained event rows**, including first rows,
so this is neither a per-transition rate nor the fraction of visits with errors.

In [18]:
step_indices = {step: idx for idx, step in enumerate(sequence)}
events["error"] = 0
grouped = events.groupby(["visit_id"])

# Iterate over each group
for _, group in grouped:
    # Get the process steps for the current group
    steps = group['process_step'].tolist()
    # Iterate over each step in the sequence
    for i in range(1, len(steps)):
        # Check if the current step is before the previous step in the sequence
        if step_indices[steps[i]] < step_indices[steps[i - 1]]:
            # Find the indices where the step goes back
            idx_current = group.index[i]
            # Update the 'error' column to 1 for the corresponding indices
            events.loc[idx_current, 'error'] = 1

In [19]:
error_summary = events.groupby("variation").error.agg(errors="sum", retained_rows="count")
error_summary["error_rate"] = error_summary.errors / error_summary.retained_rows
overall_error_rate = events.error.sum() / len(events)
print(f"Overall: {int(events.error.sum()):,} / {len(events):,} = {overall_error_rate:.4%}")
error_summary.assign(error_rate_percent=error_summary.error_rate * 100).drop(columns="error_rate").round(4)

Overall: 26,082 / 284,503 = 9.1676%


,errors,retained_rows,error_rate_percent
variation,,,
Control,9730,126858,7.6700
Test,16352,157645,10.3727


The source notebook's stored error counts are 26,082 error rows and 258,421 non-error
rows. Replaying its loop reproduces those counts. Its hard-coded expression instead
used 48,842 errors over the same 284,503 rows (17.1675%), which contradicts its own
outputs. The dynamic calculation above preserves the algorithm and row denominator,
and corrects that stale numerator. The original group-specific error rates are unchanged.

### 5.3 Visit-level completion

A visit is complete if it reaches `confirm` at least once. Build the binary outcome
once per `(variation, visit_id)` and use it for the KPI and both completion tests.
Repeated confirmations within a visit contribute only one success. No client count
is used as the completion denominator.

In [20]:
events["confirm"] = events.process_step.eq("confirm").astype(int)
visit_metrics = events.groupby(["variation", "visit_id"]).agg(
    completed=("confirm", "max"),
    retained_steps=("process_step", "size"),
).reset_index()
assert not visit_metrics.duplicated(["variation", "visit_id"]).any()
assert visit_metrics.completed.isin([0, 1]).all()

completion_summary = visit_metrics.groupby("variation").completed.agg(
    completed_visits="sum", total_visits="count"
)
completion_summary["completion_rate"] = (
    completion_summary.completed_visits / completion_summary.total_visits
)
completion_summary.assign(
    completion_rate_percent=completion_summary.completion_rate * 100
).drop(columns="completion_rate").round(4)

,completed_visits,total_visits,completion_rate_percent
variation,,,
Control,16040,32182,49.8415
Test,21724,37121,58.5221


In [21]:
completion_reconciliation = completion_summary[["completed_visits", "total_visits"]].copy()
completion_reconciliation["original_confirm_row_count"] = events.groupby("variation").confirm.sum()
completion_reconciliation["extra_confirm_rows"] = (
    completion_reconciliation.original_confirm_row_count - completion_reconciliation.completed_visits
)
completion_reconciliation["original_row_based_rate_percent"] = (
    completion_reconciliation.original_confirm_row_count / completion_reconciliation.total_visits * 100
)
completion_reconciliation["consistent_visit_rate_percent"] = completion_summary.completion_rate * 100
completion_reconciliation.round(4)

,completed_visits,total_visits,original_confirm_row_count,extra_confirm_rows,original_row_based_rate_percent,consistent_visit_rate_percent
variation,,,,,,
Control,16040,32182,16091,51,50.0000,49.8415
Test,21724,37121,21804,80,58.7376,58.5221


The earlier KPI counted confirmation rows, whereas the original z-test already used
binary completed visits. This reconciliation explains the KPI change and makes the
KPI and test populations consistent; it does not represent a change to the original
z-test's success counts.

### 5.4 Recorded duration of visits that complete

Retain the original secondary timing comparison: select visit IDs that contain a
confirmation, then sum their incoming intervals within each group and visit. This
is the **recorded span of a visit that completes**, not time to first confirmation:
it can include activity after confirmation and uses the original shared-ID grouping.
Summing group-level step means is not used as an average visit duration.

In [22]:
completed_visit_ids = events.loc[events.confirm.eq(1), "visit_id"].unique()
completed_events = events[events.visit_id.isin(completed_visit_ids)].copy()
completed_visit_durations = completed_events.groupby(["variation", "visit_id"])[
    "elapsed_since_previous_seconds"
].sum()
duration_summary = completed_visit_durations.groupby("variation").agg(
    visits="count", mean_seconds="mean", median_seconds="median"
)
duration_summary.round(2)

,visits,mean_seconds,median_seconds
variation,,,
Control,16051,383.52,264.0
Test,21729,320.42,199.0


Because the source groups completion eligibility by `visit_id` across the whole
event table, shared IDs can qualify a group-specific duration without a confirmation
in that same group. This secondary descriptive metric retains that behavior for
comparability and is separate from the binary completion dataset used for inference.
It should not be interpreted as a causal speed effect: the sets of completing visits differ.

## 6. Hypothesis testing

Use α = 0.05. Both completion tests use the same `completion_summary` from
`visit_metrics`. These are large-sample, **unadjusted visit-level tests**, which
assume independent observations. Repeated visits per client and shared visit IDs
can violate that assumption; p-values are conditional on it and are not
client-cluster-adjusted experimental evidence.

### 6.1 Is Test completion higher?

- H₀: p_test ≤ p_control
- H₁: p_test > p_control

Use a one-sided two-proportion z-test. Statistical significance alone does not
establish that the redesign meets the business threshold.

In [23]:
alpha = 0.05
test_successes = int(completion_summary.loc["Test", "completed_visits"])
control_successes = int(completion_summary.loc["Control", "completed_visits"])
test_total = int(completion_summary.loc["Test", "total_visits"])
control_total = int(completion_summary.loc["Control", "total_visits"])
p_test = test_successes / test_total
p_control = control_successes / control_total

completion_z, completion_p = proportions_ztest(
    count=[test_successes, control_successes],
    nobs=[test_total, control_total],
    alternative="larger",
)
print(f"Test: {p_test:.4%}; Control: {p_control:.4%}")
print(f"Difference: {(p_test - p_control) * 100:.4f} percentage points")
print(f"Z-statistic: {completion_z:.6f}; one-sided p-value: {completion_p:.6g}")
if completion_p < alpha:
    print("Reject H0: Test has higher visit-level completion under the test assumptions.")
else:
    print("Insufficient evidence of higher Test visit-level completion.")

Test: 58.5221%; Control: 49.8415%
Difference: 8.6806 percentage points
Z-statistic: 22.886499; one-sided p-value: 3.16633e-116
Reject H0: Test has higher visit-level completion under the test assumptions.


### 6.2 Does the improvement exceed the 5% relative threshold?

Interpret the assignment's minimum 5% increase as **relative uplift**:
`p_test ≥ 1.05 × p_control`. This differs from adding five percentage points.

- Risk ratio: RR = p_test / p_control
- H₀: RR ≤ 1.05
- H₁: RR > 1.05

Use the requested large-sample log-risk-ratio Wald test, with
SE(log RR) = √(1/test_successes − 1/test_total + 1/control_successes − 1/control_total).
Both groups must contain successes; no continuity correction is needed for these
large nonzero counts. This tests evidence of exceeding the threshold, not actual
implementation costs or profitability.

In [24]:
assert 0 < test_successes < test_total
assert 0 < control_successes < control_total
risk_ratio = p_test / p_control
relative_uplift = risk_ratio - 1
se_log_rr = np.sqrt(
    (1 / test_successes) - (1 / test_total)
    + (1 / control_successes) - (1 / control_total)
)
threshold_z = (np.log(risk_ratio) - np.log(1.05)) / se_log_rr
threshold_p = stats.norm.sf(threshold_z)

print(f"Risk ratio: {risk_ratio:.6f}")
print(f"Relative uplift: {relative_uplift * 100:.4f}%")
print(f"Z-statistic against RR = 1.05: {threshold_z:.6f}")
print(f"One-sided p-value: {threshold_p:.6g}")
if threshold_p < alpha:
    print("Under the test assumptions, Test completion is more than 5% higher in relative terms.")
    print("The redesign meets the stated completion-rate cost-effectiveness criterion.")
else:
    print("Insufficient evidence that relative uplift exceeds the required 5% threshold.")

Risk ratio: 1.174164
Relative uplift: 17.4164%
Z-statistic against RR = 1.05: 15.748925
One-sided p-value: 3.49292e-56
Under the test assumptions, Test completion is more than 5% higher in relative terms.
The redesign meets the stated completion-rate cost-effectiveness criterion.


### 6.3 Secondary timing comparison

Retain the original one-sided Welch test of the recorded duration of visits that
complete: H₀: μ_control ≤ μ_test; H₁: μ_control > μ_test. The estimand and shared-ID
limitations are described above. This exploratory test is conditional on completion,
assumes independent visits, and is not adjusted for multiple comparisons.

In [25]:
time_control = completed_visit_durations.loc["Control"]
time_test = completed_visit_durations.loc["Test"]
duration_test = stats.ttest_ind(time_control, time_test, equal_var=False, alternative="greater")
print(f"Control mean recorded span: {time_control.mean():.2f} seconds")
print(f"Test mean recorded span: {time_test.mean():.2f} seconds")
print(f"Welch t-statistic: {duration_test.statistic:.6f}; p-value: {duration_test.pvalue:.6g}")

Control mean recorded span: 383.52 seconds
Test mean recorded span: 320.42 seconds
Welch t-statistic: 13.458675; p-value: 1.734e-41


## 7. Key findings

In [26]:
kpi_summary = pd.DataFrame({
    "completion_rate_percent": completion_summary.completion_rate * 100,
    "backwards_error_rows_percent": error_summary.error_rate * 100,
    "mean_recorded_span_seconds_for_completing_visit_ids": duration_summary.mean_seconds,
})
kpi_summary.round(4)

,completion_rate_percent,backwards_error_rows_percent,mean_recorded_span_seconds_for_completing_visit_ids
variation,,,
Control,49.8415,7.6700,383.5170
Test,58.5221,10.3727,320.4151


In [27]:
print(f"Completion increased by {(p_test - p_control) * 100:.2f} percentage points "
      f"({relative_uplift * 100:.2f}% relative uplift).")
print(f"Backwards-step error rows: Control {error_summary.loc['Control', 'error_rate']:.2%}; "
      f"Test {error_summary.loc['Test', 'error_rate']:.2%}.")
print(f"Recorded span among completing visit IDs: Control {time_control.mean():.2f}s; "
      f"Test {time_test.mean():.2f}s.")

Completion increased by 8.68 percentage points (17.42% relative uplift).
Backwards-step error rows: Control 7.67%; Test 10.37%.
Recorded span among completing visit IDs: Control 383.52s; Test 320.42s.


## 8. Conclusion

The Test interface has higher visit-level completion and meets the stated 5%
relative completion-uplift criterion under the specified unadjusted tests. It also
has a higher rate of backwards-step error rows. The recorded span among completing
visit IDs is lower for Test, but this is not a measure of time to first confirmation
or proof that individual users complete faster.

The completion result supports the redesign on the assignment's success criterion,
with further investigation of backwards navigation and session identity before
using these results for a deployment decision. Repeated visits, shared IDs,
timestamp ties, incomplete-profile exclusions, and unmeasured costs limit the
interpretation. A client-aware sensitivity analysis would be needed for stronger
experimental inference; it is outside this cleanup's preserved methodology.

### Optional export examples (disabled)

These examples are retained for reference only. Normal execution does not export
processed data or create a processed-data directory.

In [28]:
# events.to_csv("web_events_cleaned.csv", index=False)
# completed_events.to_csv("completed_visit_events.csv", index=False)
# experiment_clients.to_csv("experiment_clients.csv", index=False)